In [1]:
from api_setup import setup
import re
import json

client, model_id = setup()

In [2]:
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0
    
    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunk_text = text[start_idx:end_idx]
        chunks.append(chunk_text)
        
        start_idx = (
            end_idx - chunk_overlap if end_idx < len(text) else len(text)
        )
    
    return chunks

In [3]:
def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    start_idx = 0
    
    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        current_chunk = sentences[start_idx:end_idx]
        chunks.append(" ".join(current_chunk))
        
        start_idx += max_sentences_per_chunk - overlap_sentences
        
        if start_idx < 0:
            start_idx = 0
    
    return chunks

In [4]:
def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [5]:
response = client.converse(
    modelId=model_id,
    messages=[{
        "role": "user",
        "content": [ 
            {
                "text": "write me a short medical document with clear headers for each section."
            }
        ]
    }]
)

In [6]:
document = None
with open('report.md', 'r') as f:
    document = f.read()

In [7]:
chunks = chunk_by_section(document)
print(chunks)

['# **Annual Interdisciplinary Research Review: Cross-Domain Insights**\n', "Executive Summary\n\nThis report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challenges that transcend traditional disciplinary boundaries. This year's review highlights significant progress in ten critical areas. Advances in **Medical Research** focused on the rare XDR-471 syndrome, yielding new diagnostic insights. Concurrently, **Software Engineering** tackled persistent stability issues, implementing key fixes identified through error code analysis (e.g., `ERR_MEM_ALLOC_FAIL_0x8007000E`). **Financial Analysis** revealed mixed quarterly performance, prompting strategic reviews, particularly concerning resource allocation impacting R&D pipelines.\n\nCrucial developments were also seen in **Sci

In [26]:
result2 = chunk_by_sentence(document)
print(result2)

["# **Annual Interdisciplinary Research Review: Cross-Domain Insights**\n\n## Executive Summary\n\nThis report synthesizes the key findings and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challenges that transcend traditional disciplinary boundaries. This year's review highlights significant progress in ten critical areas. Advances in **Medical Research** focused on the rare XDR-471 syndrome, yielding new diagnostic insights. Concurrently, **Software Engineering** tackled persistent stability issues, implementing key fixes identified through error code analysis (e.g., `ERR_MEM_ALLOC_FAIL_0x8007000E`).", 'Concurrently, **Software Engineering** tackled persistent stability issues, implementing key fixes identified through error code analysis (e.g., `ERR_MEM_ALLOC_FAIL_0x8007000E`). **Financial Analysis** 

In [27]:
result3 = chunk_by_char(document)
print(result3)

['# **Annual Interdisciplinary Research Review: Cross-Domain Insights**\n\n## Executive Summary\n\nThis report synthesizes the key findings and ongoing rese', "ngs and ongoing research efforts across the organization's diverse operational and R&D departments for the past fiscal year. Our strength lies in the ", 'trength lies in the cross-pollination of ideas and methodologies, driving innovation and addressing complex challenges that transcend traditional disc', "end traditional disciplinary boundaries. This year's review highlights significant progress in ten critical areas. Advances in **Medical Research** fo", 'edical Research** focused on the rare XDR-471 syndrome, yielding new diagnostic insights. Concurrently, **Software Engineering** tackled persistent st', 'ackled persistent stability issues, implementing key fixes identified through error code analysis (e.g., `ERR_MEM_ALLOC_FAIL_0x8007000E`). **Financial', '7000E`). **Financial Analysis** revealed mixed quarterly performance, 

In [9]:
def generate_embedding(
    text,
    embedding_model_id="amazon.titan-embed-text-v2:0",
    dimensions=1024,
    normalize=True,
):
    request_body = {
        "inputText": text,
        "dimensions": dimensions,
        "normalize": normalize,
    }
    
    request_json = json.dumps(request_body)
    response = client.invoke_model(
        modelId=embedding_model_id,
        body=request_json,
        accept="application/json",
        contentType="application/json",
    )
    
    response_body = json.loads(response.get("body").read())
    return response_body["embedding"]

In [10]:
# VectorIndex implementation
import math
from typing import Callable, Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn: Optional[Callable[[str], List[float]]] = None,
    ):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError(
                "Embedding function not provided during initialization."
            )
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError(
                    "Embedding function not provided for string query."
                )
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError(
                "Query must be either a string or a list of numbers."
            )

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector: List[float], document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(
        self, vec1: List[float], vec2: List[float]
    ) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

In [11]:
embeddings = [generate_embedding(chunk) for chunk in chunks]

In [12]:
store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})

In [13]:
user_embedding = generate_embedding("What did the software engineering dept do last year?")

results = store.search(user_embedding, 2)

for doc, distance in results:
    print(distance, "\n", doc["content"][0:200], "\n")

0.6907249165167906 
 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems 

0.7210594524998164 
 Methodology

The insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of findings drawn from standard departmental reporting cycles, specialized project updates 



In [40]:
user_embedding = generate_embedding("What happened with INC-2023-Q4-011?")

results = store.search(user_embedding, 2)

for doc, distance in results:
    print(distance, "\n", doc["content"][0:200], "\n")

0.6357748160533747 
 Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2 

0.8133151891807425 
 Section 3: Financial Analysis - Q3 Performance and Outlook

Quarterly financial analysis revealed a complex picture. Overall group revenue saw modest growth of 3.1% year-over-year, primarily driven by 



In [34]:
# BM25Index implementation
import math
from collections import Counter

class BM25Index:
    """BM25 lexical search returning exponentially normalized distances."""

    def __init__(self, k1=1.5, b=0.75):
        if not math.isfinite(k1) or k1 <= 0:
            raise ValueError("k1 must be finite and positive.")
        if not 0 <= b <= 1:
            raise ValueError("b must be between 0 and 1.")

        self.k1 = k1
        self.b = b
        self.documents = []
        self._term_frequencies = []
        self._document_lengths = []
        self._document_frequency = Counter()
        self._total_length = 0

    @staticmethod
    def _tokenize(text):
        # Preserve hyphenated words and identifiers.
        return re.findall(r"\w+(?:-\w+)*", text.casefold())

    def add_document(self, document):
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document must contain a 'content' key.")
        if not isinstance(document["content"], str):
            raise TypeError("Document 'content' must be a string.")

        tokens = self._tokenize(document["content"])
        frequencies = Counter(tokens)

        self.documents.append(dict(document))
        self._term_frequencies.append(frequencies)
        self._document_lengths.append(len(tokens))
        self._total_length += len(tokens)
        self._document_frequency.update(frequencies.keys())

    def search(self, query, k=1, score_normalization_factor=0.1):
        if not isinstance(query, str):
            raise TypeError("Query must be a string.")
        if not isinstance(k, int) or isinstance(k, bool) or k <= 0:
            raise ValueError("k must be a positive integer.")
        if (
            not math.isfinite(score_normalization_factor)
            or score_normalization_factor <= 0
        ):
            raise ValueError(
                "score_normalization_factor must be finite and positive."
            )

        query_terms = set(self._tokenize(query))
        count = len(self.documents)
        if not count or not query_terms or not self._total_length:
            return []

        average_length = self._total_length / count

        idf = {}
        for term in query_terms:
            df = self._document_frequency[term]
            if df:
                idf[term] = math.log1p(
                    (count - df + 0.5) / (df + 0.5)
                )

        results = []
        for index, frequencies in enumerate(self._term_frequencies):
            length = self._document_lengths[index]
            normalization = self.k1 * (
                1 - self.b + self.b * length / average_length
            )

            score = 0.0
            for term, weight in idf.items():
                frequency = frequencies[term]
                if frequency:
                    score += weight * (
                        frequency * (self.k1 + 1)
                        / (frequency + normalization)
                    )

            if score > 1e-9:
                results.append((dict(self.documents[index]), score))

        # Higher BM25 scores are better.
        results.sort(key=lambda item: item[1], reverse=True)

        # Convert scores to distances: lower is better.
        normalized_results = [
            (doc, math.exp(-score_normalization_factor * raw_score))
            for doc, raw_score in results[:k]
        ]
        normalized_results.sort(key=lambda item: item[1])
        return normalized_results

    def __len__(self):
        return len(self.documents)

In [35]:
# Create a BM25 store
store = BM25Index()

# Add documents to the store
for chunk in chunks:
    store.add_document({"content": chunk})

# Search the store
results = store.search("What happened with INC-2023-Q4-011?", 3)

for doc, distance in results:
    print(distance, "\n", doc["content"][0:200], "\n")

0.7078702514609958 
 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems 

0.7231471366352841 
 Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2 

0.9396231886697972 
 Methodology

The insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of findings drawn from standard departmental reporting cycles, specialized project updates 



In [38]:
import math
import re

import bm25s
from bm25s.tokenization import Tokenizer


class BM25Index:
    """BM25S-backed lexical search with lower-is-better distances."""

    def __init__(self, k1=1.5, b=0.75):
        if not math.isfinite(k1) or k1 <= 0:
            raise ValueError("k1 must be finite and positive.")
        if not 0 <= b <= 1:
            raise ValueError("b must be between 0 and 1.")

        self.k1 = k1
        self.b = b
        self.documents = []
        self._retriever = None
        self._tokenizer = None
        self._dirty = True

    @staticmethod
    def _tokenize(text):
        # Preserve hyphenated words and identifiers.
        return re.findall(r"\w+(?:-\w+)*", text.casefold())

    def add_document(self, document):
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document must contain a 'content' key.")
        if not isinstance(document["content"], str):
            raise TypeError("Document 'content' must be a string.")

        self.documents.append(dict(document))
        self._dirty = True

    def _build_index(self):
        self._retriever = None
        self._tokenizer = Tokenizer(
            stemmer=None,
            stopwords=[],
            splitter=self._tokenize,
        )

        texts = [doc["content"] for doc in self.documents]

        if texts and any(self._tokenize(text) for text in texts):
            corpus_tokens = self._tokenizer.tokenize(texts)
            self._retriever = bm25s.BM25(
                k1=self.k1,
                b=self.b,
                method="atire",
                idf_method="lucene",
            )
            self._retriever.index(corpus_tokens)

        self._dirty = False

    def search(self, query, k=1, score_normalization_factor=0.1):
        if not isinstance(query, str):
            raise TypeError("Query must be a string.")
        if not isinstance(k, int) or isinstance(k, bool) or k <= 0:
            raise ValueError("k must be a positive integer.")
        if (
            not math.isfinite(score_normalization_factor)
            or score_normalization_factor <= 0
        ):
            raise ValueError(
                "score_normalization_factor must be finite and positive."
            )

        if not self.documents or not self._tokenize(query):
            return []

        if self._dirty:
            self._build_index()

        if self._retriever is None:
            return []

        # Deduplicate query terms to match the original implementation.
        query = " ".join(dict.fromkeys(self._tokenize(query)))
        query_tokens = self._tokenizer.tokenize([query])

        doc_ids, scores = self._retriever.retrieve(
            query_tokens,
            k=min(k, len(self.documents)),
        )

        results = [
            (
                dict(self.documents[int(doc_id)]),
                math.exp(-score_normalization_factor * float(score)),
            )
            for doc_id, score in zip(doc_ids[0], scores[0])
            if score > 1e-9
        ]
        results.sort(key=lambda item: item[1])
        return results

    def __len__(self):
        return len(self.documents)

In [39]:
store = BM25Index()

for chunk in chunks:
    store.add_document({"content": chunk})

results = store.search("What happened with INC-2023-Q4-011?", 3)

for doc, distance in results:
    print(distance, "\n", doc["content"][:200], "\n")

0.7078702526441601 
 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems 

0.7231471321362735 
 Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2 

0.9396231846769139 
 Methodology

The insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of findings drawn from standard departmental reporting cycles, specialized project updates 

